# 04c — Features de diversidad (8 features)


In [1]:
import pandas as pd
import numpy as np
import re, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC_GUSTOS = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '02_features'
DATA_QC_GUSTOS.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None

sample = pd.read_parquet(DATA_QC_GUSTOS / 'sample_user_ids_gustos.parquet')
sample_ids = set(sample['user_id'])
N = len(sample)
features = pd.DataFrame({'user_id': sample['user_id'].values})
print(f'Sample N: {N:,}')

import pickle
from scipy.stats import entropy
t0 = time.time()

def shannon(values):
    arr = np.asarray(list(values), dtype=float)
    s = arr.sum()
    if s == 0: return np.nan
    p = arr / s
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

def gini(values):
    arr = np.sort(np.asarray(list(values), dtype=float))
    n = len(arr); s = arr.sum()
    if n == 0 or s == 0: return np.nan
    return float((2*np.sum((np.arange(1, n+1))*arr) - (n+1)*s) / (n*s))

def simpson(values):
    arr = np.asarray(list(values), dtype=float)
    s = arr.sum()
    if s == 0: return np.nan
    p = arr / s
    return float(1 - np.sum(p**2))


Sample N: 114,412


In [2]:
# entropy_char_class
print('[entropy char]')
t = time.time()
chars_df = pd.read_csv(DATA_RAW/'characters.csv', usecols=['user_id','class'], low_memory=False)
chars_df['user_id'] = chars_df['user_id'].apply(clean_id)
chars_df = chars_df[chars_df['user_id'].isin(sample_ids)]
class_counts = chars_df.groupby(['user_id','class']).size().unstack(fill_value=0)
ent = class_counts.apply(lambda row: shannon(row.values), axis=1)
features['entropy_char_class'] = features['user_id'].map(ent.to_dict())
del chars_df, class_counts
print(f'OK {time.time()-t:.1f}s')


[entropy char]


OK 4.4s


In [3]:
# entropy/simpson items_family (item_definition_excel_id como proxy de family)
print('[entropy items]... chunks')
t = time.time()
item_counts = {}  # user -> {item_def: count}
for chunk in pd.read_csv(DATA_RAW/'user_items.csv',
                         usecols=['user_id','item_definition_excel_id'],
                         chunksize=2_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    for u, g in chunk.groupby('user_id'):
        cnts = g['item_definition_excel_id'].value_counts().to_dict()
        if u not in item_counts: item_counts[u] = {}
        for k, v in cnts.items():
            item_counts[u][k] = item_counts[u].get(k,0) + v

features['entropy_items_family'] = features['user_id'].map({u: shannon(d.values()) for u, d in item_counts.items()})
features['simpson_items_family'] = features['user_id'].map({u: simpson(d.values()) for u, d in item_counts.items()})
del item_counts
print(f'OK {time.time()-t:.1f}s')


[entropy items]... chunks


OK 28.0s


In [4]:
# currency diversity from pkl
print('[currency div]...')
import pickle
t = time.time()
with open(DATA_QC_GUSTOS/'_currency_agg_concept.pkl','rb') as f:
    agg_concept = pickle.load(f)
with open(DATA_QC_GUSTOS/'_currency_agg_currency.pkl','rb') as f:
    agg_currency = pickle.load(f)

features['entropy_currency_concept'] = features['user_id'].map({u: shannon(d.values()) for u,d in agg_concept.items()})
features['gini_currency_concept'] = features['user_id'].map({u: gini(d.values()) for u,d in agg_concept.items()})
features['n_distinct_concepts_used'] = features['user_id'].map({u: int(len(d)) for u,d in agg_concept.items()})
features['entropy_currency_type'] = features['user_id'].map({u: shannon(d.values()) for u,d in agg_currency.items()})
print(f'OK {time.time()-t:.1f}s')


[currency div]...


OK 0.4s


In [5]:
# fight type entropy from pkl
print('[fights div]...')
import pickle
t = time.time()
with open(DATA_QC_GUSTOS/'_fights_type_counts.pkl','rb') as f:
    fights_counts = pickle.load(f)
features['entropy_fights_type'] = features['user_id'].map({u: shannon(d.values()) for u,d in fights_counts.items()})
print(f'OK {time.time()-t:.1f}s')


[fights div]...
OK 0.1s


In [6]:
assert len(features)==N and features['user_id'].is_unique
features.to_parquet(DATA_QC_GUSTOS / 'features_diversidad.parquet', index=False)
print(f'Guardado: {features.shape}')
nan_rates = (features.isna().mean()*100).round(2)
print(nan_rates.sort_values(ascending=False).head(10).to_string())

rep_dir = INFORMES / '04c_diversidad'
rep_dir.mkdir(parents=True, exist_ok=True)
report = [f'# 04c - Diversidad\n', f'**Shape**: {features.shape}\n', f'**Tiempo**: {time.time()-t0:.1f}s\n', '\n| Feature | %NaN |', '|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).items():
    report.append(f'| {f} | {v} |')
(rep_dir / 'execution_report.md').write_text('\n'.join(report))


Guardado: (114412, 9)
entropy_fights_type         82.01
entropy_currency_concept    81.34
gini_currency_concept       81.34
n_distinct_concepts_used    81.34
entropy_currency_type       81.34
user_id                      0.00
entropy_char_class           0.00
entropy_items_family         0.00
simpson_items_family         0.00


376